<a href="https://colab.research.google.com/github/aadityane93/Toxic_Comments_Sentiment_Analysis/blob/aaditya/1_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Toxic Comment Classification

This notebook explores the Jigsaw Toxic Comment Classification Challenge dataset and prepares the project for the modeling phase.


## Dataset Source

Dataset: Jigsaw Toxic Comment Classification Challenge  
Kaggle link: https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge

The dataset contains Wikipedia comments labeled with six toxicity categories: `toxic`, `severe_toxic`, `obscene`, `threat`, `insult`, and `identity_hate`. A comment can have more than one label, so the main target is naturally multi-label.


## Imports


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

try:
    import matplotlib.pyplot as plt
    import seaborn as sns

    PLOTTING_AVAILABLE = True
    sns.set_theme(style="whitegrid")
except Exception as exc:
    PLOTTING_AVAILABLE = False
    print(f"Plotting libraries are not available in this environment: {exc}")


Plotting libraries are not available in this environment: No module named 'matplotlib'


## Load Data


In [2]:
DATA_DIR = Path(".")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_labels = pd.read_csv(DATA_DIR / "test_labels.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]

# Keep text columns safe for later string operations.
train_df["comment_text"] = train_df["comment_text"].fillna("").astype(str)
test_df["comment_text"] = test_df["comment_text"].fillna("").astype(str)

dataframes = {
    "train": train_df,
    "test": test_df,
    "test_labels": test_labels,
    "sample_submission": sample_submission,
}

for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns")


train: 159,571 rows x 8 columns
test: 153,164 rows x 2 columns
test_labels: 153,164 rows x 7 columns
sample_submission: 153,164 rows x 7 columns


# Data Exploration


## 1. File and Column Overview

This section checks which files are present, how large they are, and which columns each file contains.


In [3]:
file_names = [
    "train.csv",
    "test.csv",
    "test_labels.csv",
    "sample_submission.csv",
]

file_overview = []
for file_name in file_names:
    path = DATA_DIR / file_name
    df_name = file_name.replace(".csv", "")
    df_key = "sample_submission" if df_name == "sample_submission" else df_name
    df = dataframes[df_key]
    file_overview.append(
        {
            "file": file_name,
            "size_mb": path.stat().st_size / (1024 * 1024),
            "rows": len(df),
            "columns": df.shape[1],
            "column_names": ", ".join(df.columns),
        }
    )

file_overview_df = pd.DataFrame(file_overview)
file_overview_df["size_mb"] = file_overview_df["size_mb"].round(2)
display(file_overview_df)


,file,size_mb,rows,columns,column_names
0,train.csv,65.6200,159571,8,"id, comment_text, toxic, severe_toxic, obscene, threat, insult, identity_hate"
1,test.csv,57.5600,153164,2,"id, comment_text"
2,test_labels.csv,4.7500,153164,7,"id, toxic, severe_toxic, obscene, threat, insult, identity_hate"
3,sample_submission.csv,5.9900,153164,7,"id, toxic, severe_toxic, obscene, threat, insult, identity_hate"


## 2. Data Quality Checks

Before modeling, we check missing values, duplicate IDs, empty comments, URL-like text, and uppercase-heavy comments. These checks help decide whether extra cleaning is needed.


In [4]:
quality_rows = []
for name, df in dataframes.items():
    row = {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "total_missing_values": int(df.isna().sum().sum()),
        "duplicate_ids": int(df["id"].duplicated().sum()) if "id" in df.columns else np.nan,
    }

    if "comment_text" in df.columns:
        text = df["comment_text"].fillna("").astype(str)
        alpha_counts = text.apply(lambda x: max(sum(ch.isalpha() for ch in x), 1))
        upper_counts = text.apply(lambda x: sum(ch.isupper() for ch in x))
        uppercase_ratio = upper_counts / alpha_counts
        row.update(
            {
                "missing_comment_text": int(df["comment_text"].isna().sum()),
                "empty_after_strip": int((text.str.strip() == "").sum()),
                "contains_newline": int(text.str.contains("\n", regex=False).sum()),
                "contains_url_like": int(text.str.contains(r"https?://|www\.", case=False, regex=True).sum()),
                "uppercase_ratio_gt_0_5": int((uppercase_ratio > 0.5).sum()),
            }
        )

    quality_rows.append(row)

quality_df = pd.DataFrame(quality_rows).fillna("-")
display(quality_df)


,dataset,rows,columns,total_missing_values,duplicate_ids,missing_comment_text,empty_after_strip,contains_newline,contains_url_like,uppercase_ratio_gt_0_5
0,train,159571,8,0,0,0.0000,0.0000,"94,466.0000","5,129.0000","2,766.0000"
1,test,153164,2,0,0,0.0000,1.0000,"85,763.0000","3,850.0000","4,356.0000"
2,test_labels,153164,7,0,0,-,-,-,-,-
3,sample_submission,153164,7,0,0,-,-,-,-,-


In [5]:
missing_by_column = pd.concat(
    {name: df.isna().sum() for name, df in dataframes.items()},
    axis=1,
).fillna("-")

missing_by_column


## 3. Label Distribution in the Training Set

The labels are highly imbalanced. Most comments are clean, while categories such as `threat`, `identity_hate`, and `severe_toxic` are rare. This matters because accuracy alone can look high even when the model misses minority classes.


In [6]:
label_counts = train_df[label_columns].sum().astype(int)
label_distribution = pd.DataFrame(
    {
        "positive_count": label_counts,
        "negative_count": len(train_df) - label_counts,
        "positive_percent": (label_counts / len(train_df) * 100).round(4),
        "negative_to_positive_ratio": ((len(train_df) - label_counts) / label_counts).round(2),
    }
).sort_values("positive_count", ascending=False)

display(label_distribution)


,positive_count,negative_count,positive_percent,negative_to_positive_ratio
toxic,15294,144277,9.5844,9.4300
obscene,8449,151122,5.2948,17.8900
insult,7877,151694,4.9364,19.2600
severe_toxic,1595,157976,0.9996,99.0400
identity_hate,1405,158166,0.8805,112.5700
threat,478,159093,0.2996,332.8300


In [7]:
if PLOTTING_AVAILABLE:
    ax = label_distribution.sort_values("positive_count")["positive_count"].plot(
        kind="barh",
        figsize=(8, 4),
        color="#4C78A8",
    )
    ax.set_title("Positive Label Counts in train.csv")
    ax.set_xlabel("Number of positive comments")
    ax.set_ylabel("Label")
    plt.tight_layout()
    plt.show()
else:
    display(label_distribution[["positive_count", "positive_percent"]])


,positive_count,positive_percent
toxic,15294,9.5844
obscene,8449,5.2948
insult,7877,4.9364
severe_toxic,1595,0.9996
identity_hate,1405,0.8805
threat,478,0.2996


## 4. Binary Toxicity View

For the binary task, we create `any_toxic`, where a comment is toxic if at least one of the six label columns is positive.


In [8]:
train_df["label_count"] = train_df[label_columns].sum(axis=1)
train_df["any_toxic"] = (train_df["label_count"] > 0).astype(int)
train_df["toxicity_group"] = np.where(train_df["any_toxic"] == 1, "toxic", "clean")

binary_distribution = train_df["toxicity_group"].value_counts().rename_axis("group").reset_index(name="count")
binary_distribution["percent"] = (binary_distribution["count"] / len(train_df) * 100).round(4)

display(binary_distribution)


,group,count,percent
0,clean,143346,89.8321
1,toxic,16225,10.1679


In [9]:
if PLOTTING_AVAILABLE:
    ax = binary_distribution.plot(
        x="group",
        y="count",
        kind="bar",
        legend=False,
        figsize=(5, 4),
        color=["#72B7B2", "#E45756"],
    )
    ax.set_title("Clean vs Any Toxic Comments")
    ax.set_xlabel("")
    ax.set_ylabel("Number of comments")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    display(binary_distribution)


,group,count,percent
0,clean,143346,89.8321
1,toxic,16225,10.1679


## 5. Multi-label Structure

The dataset is multi-label because one comment can belong to several toxicity categories. This section checks how many labels each comment has and which label combinations are most common.


In [10]:
labels_per_comment = (
    train_df["label_count"]
    .value_counts()
    .sort_index()
    .rename_axis("number_of_active_labels")
    .reset_index(name="comment_count")
)
labels_per_comment["percent"] = (labels_per_comment["comment_count"] / len(train_df) * 100).round(4)

display(labels_per_comment)


,number_of_active_labels,comment_count,percent
0,0,143346,89.8321
1,1,6360,3.9857
2,2,3480,2.1808
3,3,4209,2.6377
4,4,1760,1.1030
5,5,385,0.2413
6,6,31,0.0194


In [11]:
def active_label_names(row):
    active = [label for label in label_columns if row[label] == 1]
    return "clean" if len(active) == 0 else "+".join(active)

train_df["active_labels"] = train_df[label_columns].apply(active_label_names, axis=1)

top_label_combinations = (
    train_df["active_labels"]
    .value_counts()
    .head(15)
    .rename_axis("label_combination")
    .reset_index(name="comment_count")
)
top_label_combinations["percent"] = (top_label_combinations["comment_count"] / len(train_df) * 100).round(4)

display(top_label_combinations)


,label_combination,comment_count,percent
0,clean,143346,89.8321
1,toxic,5666,3.5508
2,toxic+obscene+insult,3800,2.3814
3,toxic+obscene,1758,1.1017
4,toxic+insult,1215,0.7614
5,toxic+severe_toxic+obscene+insult,989,0.6198
6,toxic+obscene+insult+identity_hate,618,0.3873
7,obscene,317,0.1987
8,insult,301,0.1886
9,toxic+severe_toxic+obscene+insult+identity_hate,265,0.1661


In [12]:
co_occurrence = train_df[label_columns].T.dot(train_df[label_columns]).astype(int)
co_occurrence


In [13]:
if PLOTTING_AVAILABLE:
    plt.figure(figsize=(7, 5))
    sns.heatmap(co_occurrence, annot=True, fmt="d", cmap="Blues")
    plt.title("Label Co-occurrence Counts")
    plt.tight_layout()
    plt.show()
else:
    display(co_occurrence)


,toxic,severe_toxic,obscene,threat,insult,identity_hate
toxic,15294,1595,7926,449,7344,1302
severe_toxic,1595,1595,1517,112,1371,313
obscene,7926,1517,8449,301,6155,1032
threat,449,112,301,478,307,98
insult,7344,1371,6155,307,7877,1160
identity_hate,1302,313,1032,98,1160,1405


## 6. Comment Length Exploration

Comment length can affect preprocessing choices such as maximum sequence length for neural networks. We compare character and word lengths for the train and test sets, and for clean vs toxic training comments.


In [14]:
for df in [train_df, test_df]:
    df["char_count"] = df["comment_text"].str.len()
    df["word_count"] = df["comment_text"].str.split().str.len()

length_summary_rows = []
for name, df in {"train": train_df, "test": test_df}.items():
    for column in ["char_count", "word_count"]:
        length_summary_rows.append(
            {
                "dataset": name,
                "metric": column,
                "mean": df[column].mean(),
                "median": df[column].median(),
                "p90": df[column].quantile(0.90),
                "p95": df[column].quantile(0.95),
                "max": df[column].max(),
            }
        )

length_summary = pd.DataFrame(length_summary_rows)
for column in ["mean", "median", "p90", "p95", "max"]:
    length_summary[column] = length_summary[column].round(2)

display(length_summary)


,dataset,metric,mean,median,p90,p95,max
0,train,char_count,394.0700,205.0000,889.0000,"1,355.0000",5000
1,train,word_count,67.2700,36.0000,152.0000,230.0000,1411
2,test,char_count,364.8800,180.0000,804.0000,"1,273.8500",5000
3,test,word_count,61.6100,31.0000,136.0000,213.0000,2321


In [15]:
length_by_toxicity = (
    train_df
    .groupby("toxicity_group")[["char_count", "word_count"]]
    .agg(["mean", "median", "max"])
    .round(2)
)

length_by_toxicity


In [16]:
if PLOTTING_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    train_df["char_count"].clip(upper=2000).plot(kind="hist", bins=50, ax=axes[0], color="#4C78A8")
    axes[0].set_title("Train Comment Lengths, Clipped at 2000 Characters")
    axes[0].set_xlabel("Character count")

    train_df["word_count"].clip(upper=400).plot(kind="hist", bins=50, ax=axes[1], color="#F58518")
    axes[1].set_title("Train Comment Lengths, Clipped at 400 Words")
    axes[1].set_xlabel("Word count")

    plt.tight_layout()
    plt.show()
else:
    display(length_summary)


,dataset,metric,mean,median,p90,p95,max
0,train,char_count,394.0700,205.0000,889.0000,"1,355.0000",5000
1,train,word_count,67.2700,36.0000,152.0000,230.0000,1411
2,test,char_count,364.8800,180.0000,804.0000,"1,273.8500",5000
3,test,word_count,61.6100,31.0000,136.0000,213.0000,2321


## 7. Test Labels and `-1` Values

The test label file contains `-1` values for rows that were not used in the original competition scoring. These rows should be removed when evaluating on `test_labels.csv`.


In [17]:
valid_test_mask = (test_labels[label_columns] != -1).all(axis=1)
valid_test_labels = test_labels.loc[valid_test_mask].copy()

validity_summary = pd.DataFrame(
    {
        "group": ["valid labeled rows", "rows containing -1"],
        "count": [int(valid_test_mask.sum()), int((~valid_test_mask).sum())],
    }
)
validity_summary["percent"] = (validity_summary["count"] / len(test_labels) * 100).round(4)

display(validity_summary)


,group,count,percent
0,valid labeled rows,63978,41.7709
1,rows containing -1,89186,58.2291


In [18]:
valid_test_label_counts = valid_test_labels[label_columns].sum().astype(int)
valid_test_distribution = pd.DataFrame(
    {
        "positive_count": valid_test_label_counts,
        "positive_percent": (valid_test_label_counts / len(valid_test_labels) * 100).round(4),
    }
).sort_values("positive_count", ascending=False)

display(valid_test_distribution)


,positive_count,positive_percent
toxic,6090,9.5189
obscene,3691,5.7692
insult,3427,5.3565
identity_hate,712,1.1129
severe_toxic,367,0.5736
threat,211,0.3298


In [19]:
valid_test_binary = (valid_test_labels[label_columns].sum(axis=1) > 0).map({True: "toxic", False: "clean"})
valid_test_binary_distribution = valid_test_binary.value_counts().rename_axis("group").reset_index(name="count")
valid_test_binary_distribution["percent"] = (valid_test_binary_distribution["count"] / len(valid_test_labels) * 100).round(4)

display(valid_test_binary_distribution)


,group,count,percent
0,clean,57735,90.2420
1,toxic,6243,9.7580


## 8. Ethical Handling of Text Examples

This dataset contains offensive and harmful language. For the final report, it is better to discuss examples at a high level instead of quoting raw toxic comments. The cell below shows metadata-only samples without displaying the actual comment text.


In [20]:
sample_metadata = (
    train_df[["id", "toxicity_group", "label_count", "active_labels", "char_count", "word_count"]]
    .sample(10, random_state=42)
    .sort_values(["toxicity_group", "label_count"], ascending=[False, False])
)

display(sample_metadata)


,id,toxicity_group,label_count,active_labels,char_count,word_count
119105,7ca72b5b9c688e9e,clean,0,clean,325,62
131631,c03f72fd8f8bf54f,clean,0,clean,232,41
125326,9e5b8e8fc1ff2e84,clean,0,clean,65,12
111256,5332799e706665a6,clean,0,clean,464,66
83590,dfa7d8f0b4366680,clean,0,clean,117,22
37546,64479b84de1d00c1,clean,0,clean,608,115
98371,0e3561a3ab12ebee,clean,0,clean,36,6
67118,b393676802817dac,clean,0,clean,39,5
129625,b5632fa10019dbdc,clean,0,clean,454,82
48941,82d99700af45e2a8,clean,0,clean,59,10


## 9. Report-ready EDA Summary

Run the cell below after the EDA cells. It generates a short dataset exploration paragraph that can be adapted for the written report.


In [21]:
most_common_label = label_distribution["positive_count"].idxmax()
rarest_label = label_distribution["positive_count"].idxmin()
clean_count = int((train_df["any_toxic"] == 0).sum())
toxic_count = int((train_df["any_toxic"] == 1).sum())
multi_label_count = int((train_df["label_count"] > 1).sum())
valid_test_rows = int(valid_test_mask.sum())
excluded_test_rows = int((~valid_test_mask).sum())

summary_text = f"""
### Dataset Exploration Summary

The Jigsaw Toxic Comment Classification dataset contains {len(train_df):,} training comments and {len(test_df):,} test comments. The training file includes one text column and six binary toxicity labels: {', '.join(label_columns)}. There are no missing values or duplicate IDs in the training set. The dataset is strongly imbalanced: {clean_count:,} training comments ({clean_count / len(train_df) * 100:.2f}%) are clean, while {toxic_count:,} comments ({toxic_count / len(train_df) * 100:.2f}%) have at least one toxicity label. The most frequent toxicity label is `{most_common_label}`, and the rarest label is `{rarest_label}`. Multi-label behavior is important because {multi_label_count:,} comments have more than one active toxicity label. The test label file contains {valid_test_rows:,} valid labeled rows and {excluded_test_rows:,} rows with `-1` values, which should be excluded during evaluation. These findings show that the project should use metrics such as precision, recall, F1 score, ROC-AUC, and PR-AUC rather than relying only on accuracy.
"""

display(Markdown(summary_text))



### Dataset Exploration Summary

The Jigsaw Toxic Comment Classification dataset contains 159,571 training comments and 153,164 test comments. The training file includes one text column and six binary toxicity labels: toxic, severe_toxic, obscene, threat, insult, identity_hate. There are no missing values or duplicate IDs in the training set. The dataset is strongly imbalanced: 143,346 training comments (89.83%) are clean, while 16,225 comments (10.17%) have at least one toxicity label. The most frequent toxicity label is `toxic`, and the rarest label is `threat`. Multi-label behavior is important because 9,865 comments have more than one active toxicity label. The test label file contains 63,978 valid labeled rows and 89,186 rows with `-1` values, which should be excluded during evaluation. These findings show that the project should use metrics such as precision, recall, F1 score, ROC-AUC, and PR-AUC rather than relying only on accuracy.


# Next Step

After EDA, the next step is to define the two project tasks clearly:

1. Binary classification: clean vs any toxic comment.
2. Multi-label classification: predict the six toxicity categories.

The class imbalance found during EDA should guide the modeling choices, especially the use of weighted loss functions and F1-based evaluation.
